In [1]:
# Incremental Merkle tree
from collections import defaultdict

ZERO = 0
DEPTH = 3

def hash(a, b):
    return f'h({a},{b})'

class MerkleTree:
    def __init__(self):
        # Next leaf index = current leaf count
        self.next = 0
        self.root = ""
        # Merkle root of all zeros
        self.zero_root = ""
        self.tree_num = 0
        
        # Zero hash at each level
        self.zeros = [ZERO] * DEPTH
        # Right-most left elements at each level, whose right branch is empty
        self.nodes = [ZERO] * DEPTH

        # tree num -> root -> bool
        self.hist = defaultdict(lambda: defaultdict(bool))

        ## Debug ##
        # Leaves
        self.vals = [ZERO] * (2**DEPTH)

        # Calculate tree with all leaves = ZERO
        z = ZERO
        for i in range(DEPTH):
            self.zeros[i] = z
            self.nodes[i] = z
            # Next level
            z = hash(z, z)

        self.root = z
        self.new_root = z
        self.hist[0][z] = True

    # New tree
    def new(self):
        self.root = self.zero_root
        self.next = 0
        self.tree_num += 1
        self.vals = [ZERO] * (2**DEPTH)

    def insert(self, vals):
        count = len(vals)
        if count == 0:
            return

        # Create new tree if current count + insert count > 2**DEPTH
        if self.next + count > 2 ** DEPTH:
            self.new()

        # Debug (store vals)
        for i in range(count):
            self.vals[self.next + i] = vals[i]
        
        # Starting index at each level (updated at each level)
        s = self.next
        self.next += count

        # Copy vals
        hs = vals[:]

        # Next level hash index
        h = 0
        # Next level start index (parent)
        p = 0
        for level in range(DEPTH):
            p = s >> 1
            # Current level insertion index
            i = 0
            
            # Align starting index to left branch
            if s % 2 == 1:
                # h always = 0
                h = (s >> 1) - p
                assert h == 0
                # nodes[level] = right most left node
                # hs[h] = hash for next level
                hs[h] = hash(self.nodes[level], hs[i])
                i += 1
                s += 1

            # Index aligned to left branch
            # i starts at left branch
            # i jumps by 2
            # Therefore hs[i] is always the left branch
            while i < count:
                # count is even -> last i = count - 2 -> left = hs[i], right = hs[i + 1]
                # count is odd  -> last i = count - 1 -> left = hs[i], right = zero
                r = hs[i + 1] if i < count - 1 else self.zeros[level]

                # count is even -> last i = count - 2 (hs[i] = left branch)
                #          odd  -> last i = count - 1 (hs[i] = left branch)
                if i == count - 1 or i == count - 2:
                    # nodes[level] = right most left branch
                    self.nodes[level] = hs[i]

                # Calculate next level index of hs = s // 2 - next level start index
                h = (s >> 1) - p
                hs[h] = hash(hs[i], r)
                
                s += 2
                i += 2

            # Update starting index for next level
            s = p
            # Count for next level = last hash index for next level + 1
            count = h + 1

        self.root = hs[0]
        self.hist[self.tree_num][self.root] = True


In [2]:
import random

def test():
    tree = MerkleTree()
    tree.insert([1])
    tree.insert([2, 3, 4])
    tree.insert([5, 6])
    
    print(tree.root)
    print(tree.zeros)
    print(tree.nodes)
    print(tree.vals)

    tree.insert([7, 8, 9, 10])
    print(tree.root)
    print(tree.zeros)
    print(tree.nodes)
    print(tree.vals)

for _ in range(1):
    test()

h(h(h(1,2),h(3,4)),h(h(5,6),h(0,0)))
[0, 'h(0,0)', 'h(h(0,0),h(0,0))']
[5, 'h(5,6)', 'h(h(1,2),h(3,4))']
[1, 2, 3, 4, 5, 6, 0, 0]
h(h(h(7,8),h(9,10)),h(h(0,0),h(0,0)))
[0, 'h(0,0)', 'h(h(0,0),h(0,0))']
[9, 'h(7,8)', 'h(h(7,8),h(9,10))']
[7, 8, 9, 10, 0, 0, 0, 0]
